آگهی های دارای قیمت سال ساخت و یا مساحت غیر عادی را شناسایی کنید

## 17. تشخیص و مدیریت Outlier

حذف همه مقادیر خارج از سه انحراف معیار مجاز نیست. توزیع قیمت و مساحت معمولاً چوله است
و دامنه طبیعی آن به شهر، محله، نوع ملک و رژیم قیمت وابسته است.

قواعد Outlier باید حداقل این موارد را در نظر بگیرند:

- نوع ملک
- شهر و در صورت کفایت نمونه، محله
- رژیم قیمت
- مساحت بنا یا زمین
- سال ساخت
- قیمت کل
- قیمت هر مترمربع
- وضعیت توافقی یا مقطوع

روش‌های قابل استفاده:

- قواعد دامنه کسب‌وکاری
- Quantileهای گروهی
- IQR گروهی
- Log transform
- Robust Z-score
- مقایسه محلی
- Flag کردن به‌جای حذف قطعی

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

<div dir="rtl" align="right">

## Outlier Policy — Numeric Normalization, Flagging and Audit
<div dir="rtl" align="right">
در این مرحله بدون حذف هیچ رکوردی، ستون‌های عددی موردنیاز برای شناسایی داده‌های پرت آماده می‌شوند.
<div dir="rtl" align="right">  
قواعد اصلی:

- ستون‌های خام تغییر نمی‌کنند.
- برای تحلیل Outlier از ستون‌های استانداردشده استفاده می‌شود.
- قیمت فروش، رهن و اجاره بر اساس رژیم قیمت جدا بررسی می‌شوند.
- مساحت و سال ساخت جداگانه Flag می‌شوند.
- هیچ داده‌ای حذف نمی‌شود؛ فقط Flag و Reason ثبت می‌شود.
- خروجی‌های Audit برای گزارش و Power BI ذخیره می‌شوند.

<div dir="rtl" align="right">
</div>

In [2]:
# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)
# pd.set_option("display.width", None)
# pd.set_option("display.max_colwidth", None)

In [3]:
df = pd.read_feather("../Outputs/18_df.feather")

In [4]:
df.columns

Index(['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug',
       'created_at_month', 'user_type', 'description', 'title', 'rent_mode',
       'rent_value', 'rent_to_single', 'rent_type', 'price_mode',
       'price_value', 'credit_mode', 'credit_value', 'rent_credit_transform',
       'transformable_price', 'transformable_credit', 'transformed_credit',
       'transformable_rent', 'transformed_rent', 'land_size', 'building_size',
       'deed_type', 'has_business_deed', 'floor', 'rooms_count',
       'total_floors_count', 'unit_per_floor', 'has_balcony', 'has_elevator',
       'has_warehouse', 'has_parking', 'construction_year', 'is_rebuilt',
       'has_water', 'has_warm_water_provider', 'has_electricity', 'has_gas',
       'has_heating_system', 'has_cooling_system', 'has_restroom',
       'has_security_guard', 'has_barbecue', 'building_direction', 'has_pool',
       'has_jacuzzi', 'has_sauna', 'floor_material', 'property_type',
       'regular_person_capacity', 'extra_person

In [5]:
import numpy as np
import pandas as pd

price_cols = [
    'sale_price_per_sqm',
    'equivalent_monthly_rent',
    'equivalent_deposit'
]

group_cols = ['city_slug', 'cat2_slug','neighborhood_slug']

for col in price_cols:

    # تبدیل به عدد
    df[col] = pd.to_numeric(df[col], errors='coerce')

    # log
    log_col = f'{col}_log'
    df[log_col] = np.log1p(df[col].clip(lower=0))

    # Q1 و Q3 گروهی
    q1 = df.groupby(group_cols)[log_col].transform('quantile', 0.25)
    q3 = df.groupby(group_cols)[log_col].transform('quantile', 0.75)

    # IQR
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    # Flag
    df[f'{col}_outlier'] = (
        (df[log_col] < lower) |
        (df[log_col] > upper)
    )

    # مقادیر صفر یا منفی هم Flag شوند
    df.loc[df[col] <= 0, f'{col}_outlier'] = True

    # NaN را فعلاً Outlier نکن
    df.loc[df[col].isna(), f'{col}_outlier'] = False

    df.drop(columns=log_col, inplace=True)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_12968\2871054112.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  q1 = df.groupby(group_cols)[log_col].transform('quantile', 0.25)
C:\Users\lenovo\AppData\Local\Temp\ipykernel_12968\2871054112.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  q3 = df.groupby(group_cols)[log_col].transform('quantile', 0.75)
C:\Users\lenovo\AppData\Local\Temp\ipykernel_12968\2871054112.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to ad

In [6]:
outlier_cols = [
    'sale_price_per_sqm_outlier',
    'equivalent_monthly_rent_outlier',
    'equivalent_deposit_outlier'
]

outliers = df[df[outlier_cols].any(axis=1)]
outliers[
    [
        'city_slug',
        'cat2_slug',
        'sale_price_per_sqm',
        'equivalent_monthly_rent',
        'equivalent_deposit',
        'sale_price_per_sqm_outlier',
        'equivalent_monthly_rent_outlier',
        'equivalent_deposit_outlier'
    ]
]

,city_slug,cat2_slug,sale_price_per_sqm,equivalent_monthly_rent,equivalent_deposit,sale_price_per_sqm_outlier,equivalent_monthly_rent_outlier,equivalent_deposit_outlier
68,tehran,commercial-rent,NaN,100001.0,NaN,False,True,False
92,tehran,residential-rent,NaN,NaN,1.200000e+09,False,False,True
197,karaj,residential-rent,NaN,11000000.0,3.666667e+08,False,True,True
338,mashhad,residential-rent,NaN,36111110.0,1.203704e+09,False,True,True
636,ahvaz,residential-rent,NaN,2800000.0,9.333333e+07,False,False,True
...,...,...,...,...,...,...,...,...
999808,karaj,residential-rent,NaN,250000.0,8.333333e+06,False,True,True
999832,rasht,commercial-rent,NaN,1000000.0,NaN,False,True,False
999906,tehran,commercial-rent,NaN,3100000.0,1.033333e+08,False,True,True
999916,karaj,commercial-rent,NaN,190000000.0,6.333333e+09,False,False,True


In [7]:
group_cols = [
    'city_slug',
    'cat2_slug',
    'neighborhood_slug'
]

group_size = df.groupby(
    group_cols,
    observed=True
)['building_size'].transform('count')

df['building_size_log'] = np.log1p(
    df['building_size'].clip(lower=0)
)

q1 = df.groupby(
    group_cols,
    observed=True
)['building_size_log'].transform('quantile', 0.25)

q3 = df.groupby(
    group_cols,
    observed=True
)['building_size_log'].transform('quantile', 0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df['building_size_outlier'] = False

valid_group = group_size >= 20

df.loc[valid_group, 'building_size_outlier'] = (
    (df.loc[valid_group, 'building_size_log'] < lower[valid_group])
    |
    (
        (df.loc[valid_group, 'building_size_log'] > upper[valid_group])
        &
        (df.loc[valid_group, 'building_size'] >= 200)
    )
)

df.loc[df['building_size'] <= 0, 'building_size_outlier'] = True

df.loc[df['building_size'].isna(), 'building_size_outlier'] = False

df.drop(columns=['building_size_log'], inplace=True)

In [8]:
df['building_size_outlier'].sum()



np.int64(19284)

In [9]:
building_outliers = df[
    df['building_size_outlier']
]

building_outliers[
    [
        'city_slug',
        'neighborhood_slug',
        'cat2_slug',
        'building_size',
        'location_latitude',
        'location_longitude'
    ]
].head(50)

,city_slug,neighborhood_slug,cat2_slug,building_size,location_latitude,location_longitude
14,ahvaz,pardis-ahvaz,residential-sell,500.0,31.265947,48.568604
16,tehran,shahrak-naft-district5,residential-sell,700.0,35.890186,51.280197
22,mashhad,daneshjoo,residential-sell,13000.0,NaN,NaN
95,mashhad,emamreza,temporary-rent,850.0,NaN,NaN
117,shiraz,mianrood,residential-rent,40.0,NaN,NaN
176,mashhad,eqbal,residential-sell,1000.0,36.331398,59.471451
208,mashhad,bolvartoos,residential-sell,850.0,36.623306,59.501808
225,karaj,kamalshahr,residential-sell,1912.0,NaN,NaN
257,mashhad,bolvartoos,residential-sell,900.0,36.481182,59.470615
260,mashhad,sayyadshirazi,residential-sell,450.0,36.327545,59.479321


In [10]:
df.columns

Index(['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug',
       'created_at_month', 'user_type', 'description', 'title', 'rent_mode',
       'rent_value', 'rent_to_single', 'rent_type', 'price_mode',
       'price_value', 'credit_mode', 'credit_value', 'rent_credit_transform',
       'transformable_price', 'transformable_credit', 'transformed_credit',
       'transformable_rent', 'transformed_rent', 'land_size', 'building_size',
       'deed_type', 'has_business_deed', 'floor', 'rooms_count',
       'total_floors_count', 'unit_per_floor', 'has_balcony', 'has_elevator',
       'has_warehouse', 'has_parking', 'construction_year', 'is_rebuilt',
       'has_water', 'has_warm_water_provider', 'has_electricity', 'has_gas',
       'has_heating_system', 'has_cooling_system', 'has_restroom',
       'has_security_guard', 'has_barbecue', 'building_direction', 'has_pool',
       'has_jacuzzi', 'has_sauna', 'floor_material', 'property_type',
       'regular_person_capacity', 'extra_person

In [11]:
# ============================================================
# 4. IQR Outlier Function
# ============================================================

def iqr_bounds(series):
    
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    
    iqr = q3 - q1
    
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    return lower, upper

In [12]:
# ============================================================
# 5. Price Outliers
# ============================================================

df["outlier_price_flag"] = False

price_columns = [
    "sale_price_per_sqm",
    "equivalent_monthly_rent",
    "equivalent_deposit"
]

for col in price_columns:
    
    for regime, group in df.groupby("price_regime"):
        
        values = group[col].dropna()
        
        if len(values) < 10:
            continue
        
        lower, upper = iqr_bounds(values)
        
        mask = (
            (df["price_regime"] == regime) &
            (df[col] < lower) |
            (df["price_regime"] == regime) &
            (df[col] > upper)
        )
        
        df.loc[mask, "outlier_price_flag"] = True

In [13]:
# ============================================================
# 6. Area Outliers
# ============================================================

df["outlier_area_flag"] = False

values = df["building_size"].dropna()

lower_area, upper_area = iqr_bounds(values)

df.loc[
    (df["building_size"] < lower_area) |
    (df["building_size"] > upper_area),
    "outlier_area_flag"
] = True

print("Area lower bound:", lower_area)
print("Area upper bound:", upper_area)

Area lower bound: -60.0
Area upper bound: 300.0


In [14]:
df.columns

Index(['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug',
       'created_at_month', 'user_type', 'description', 'title', 'rent_mode',
       'rent_value', 'rent_to_single', 'rent_type', 'price_mode',
       'price_value', 'credit_mode', 'credit_value', 'rent_credit_transform',
       'transformable_price', 'transformable_credit', 'transformed_credit',
       'transformable_rent', 'transformed_rent', 'land_size', 'building_size',
       'deed_type', 'has_business_deed', 'floor', 'rooms_count',
       'total_floors_count', 'unit_per_floor', 'has_balcony', 'has_elevator',
       'has_warehouse', 'has_parking', 'construction_year', 'is_rebuilt',
       'has_water', 'has_warm_water_provider', 'has_electricity', 'has_gas',
       'has_heating_system', 'has_cooling_system', 'has_restroom',
       'has_security_guard', 'has_barbecue', 'building_direction', 'has_pool',
       'has_jacuzzi', 'has_sauna', 'floor_material', 'property_type',
       'regular_person_capacity', 'extra_person

In [15]:
df.to_feather("../Outputs/19_df.feather")